# Raw Image Display (No Preprocessing)

This notebook displays the same images analyzed in SAM_Visualization_Fixed.ipynb
but WITHOUT any preprocessing:
- ❌ No ToCanonical
- ❌ No ResizeLargestTo
- ❌ No CropOrPad
- ❌ No ZNormalization

Just the raw NIFTI volumes as they are!

In [ ]:
import sys
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import yaml
import random

In [ ]:
# Add project root to path
project_root = Path.cwd().resolve()
while project_root.parent != project_root and not (project_root / 'configs').exists():
    project_root = project_root.parent

if not (project_root / 'configs').exists():
    raise FileNotFoundError("Unable to locate project root with 'configs' directory")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")

## Configuration

In [ ]:
# Load datasets config (prefer visualization config)
config_candidates = [
    project_root / "configs" / "datasets_analysis.yaml",
    project_root / "configs" / "datasets.yaml"
]

config = None
config_path = None
for candidate in config_candidates:
    if candidate.exists():
        with open(candidate) as f:
            config = yaml.safe_load(f)
        config_path = candidate
        break

if config is None:
    raise FileNotFoundError("Unable to locate datasets configuration file")

# Locate SAM-Med3D root
sam3d_root = project_root / "SAM-Med3D-main" / "SAM-Med3D-main"
if not sam3d_root.exists():
    sam3d_root = project_root / "SAM-Med3D-main"

vis_config = config.get("analysis", {}).get("visualization", {})

print(f"Using config: {config_path}")
print(f"SAM-Med3D root: {sam3d_root}")
print(f"Available datasets: {list(config['datasets'].keys())}")

In [ ]:
# Settings (can be overridden via config.analysis.visualization)
n_samples_per_dataset = vis_config.get('n_samples_per_dataset', 2)
n_coords_per_sample = vis_config.get('coords_per_sample', vis_config.get('slices_per_sample', 3))
random.seed(vis_config.get('seed', 42))
np.random.seed(vis_config.get('seed', 42))

## Helper Functions

In [ ]:
def find_dataset_images(dataset_config, sam3d_root, project_root):
    """Find image/label pairs for a dataset across processed and raw locations."""
    images, labels = [], []
    seen_images = set()
    category = dataset_config.get('category')
    ct_name = dataset_config.get('ct_name', '')

    def add_pair(img_path, label_path=None):
        if img_path is None:
            return
        img_path = Path(img_path)
        if not img_path.exists():
            return
        if img_path in seen_images:
            return
        seen_images.add(img_path)
        images.append(img_path)
        labels.append(Path(label_path) if label_path and Path(label_path).exists() else None)

    # Processed SAM-Med3D layout
    processed_paths = [
        (sam3d_root / 'data' / 'train' / category / ct_name / 'imagesTr',
         sam3d_root / 'data' / 'train' / category / ct_name / 'labelsTr'),
        (sam3d_root / 'data' / 'validation' / category / ct_name / 'imagesVal',
         sam3d_root / 'data' / 'validation' / category / ct_name / 'labelsVal'),
    ]

    for img_dir, lbl_dir in processed_paths:
        if not img_dir.exists():
            continue
        label_dir_exists = lbl_dir.exists()
        for img_file in sorted(img_dir.glob('*.nii.gz')):
            label_file = None
            if label_dir_exists:
                candidate = lbl_dir / img_file.name
                if candidate.exists():
                    label_file = candidate
            add_pair(img_file, label_file)

    # Raw data layout under project_root/data
    data_root = project_root / 'data' / category if category else None
    if data_root and data_root.exists():
        case_dirs = sorted(data_root.glob('*'))
        for case_dir in case_dirs:
            nifti_dir = case_dir / '1' / 'NIFTI'
            if not nifti_dir.exists():
                nifti_dir = case_dir / 'NIFTI'
            if not nifti_dir.exists():
                continue

            img_files = list(nifti_dir.glob('image.nii.gz'))
            if not img_files:
                img_files = list(nifti_dir.glob('image_lesion_*.nii.gz'))
            if not img_files:
                img_files = list(nifti_dir.glob('image_*.nii.gz'))
            if not img_files:
                img_files = list(nifti_dir.glob('*image*.nii.gz'))
            if not img_files:
                continue

            seg_files = list(nifti_dir.glob('segmentation.nii.gz'))
            if not seg_files:
                seg_files = list(nifti_dir.glob('segmentation_lesion_*.nii.gz'))
            if not seg_files:
                seg_files = list(nifti_dir.glob('segmentation_*.nii.gz'))
            if not seg_files:
                seg_files = list(nifti_dir.glob('*label*.nii.gz'))

            add_pair(img_files[0], seg_files[0] if seg_files else None)

    # Cluster layout (if running on cluster scratch)
    cluster_paths = [
        (Path(f"/data/scratch/r112276/{category}/{ct_name}/imagesTr"),
         Path(f"/data/scratch/r112276/{category}/{ct_name}/labelsTr")),
    ]

    for img_dir, lbl_dir in cluster_paths:
        if not img_dir.exists():
            continue
        label_dir_exists = lbl_dir.exists()
        for img_file in sorted(img_dir.glob('*.nii.gz')):
            label_file = None
            if label_dir_exists:
                candidate = lbl_dir / img_file.name
                if candidate.exists():
                    label_file = candidate
            add_pair(img_file, label_file)

    return images, labels

In [ ]:
def load_raw_volume(img_path):
    """
    Load COMPLETELY RAW volume - NO PREPROCESSING AT ALL!
    
    Just reads the NIFTI file and returns the numpy array and spacing information.
    
    Returns:
        tuple: (volume, spacing) where spacing is (z_spacing, y_spacing, x_spacing) in mm
    """
    sitk_img = sitk.ReadImage(str(img_path))
    volume = sitk.GetArrayFromImage(sitk_img)
    
    # Get spacing in mm (x, y, z order from SimpleITK, need to reverse for numpy convention)
    spacing_sitk = sitk_img.GetSpacing()  # (x, y, z)
    spacing = (spacing_sitk[2], spacing_sitk[1], spacing_sitk[0])  # Convert to (z, y, x)
    
    # Apply CT clamping ONLY for better visualization (optional)
    if "/ct_" in str(img_path).lower() or "_ct" in str(img_path).lower():
        volume = np.clip(volume, -1000, 1000)
    
    return volume, spacing

In [ ]:
def load_raw_mask(mask_path):
    """Load raw mask without preprocessing."""
    sitk_mask = sitk.ReadImage(str(mask_path))
    mask = sitk.GetArrayFromImage(sitk_mask)
    mask = (mask > 0).astype(float)
    return mask

## Visualization Function

In [ ]:
def select_content_indices(volume, axis, n_indices, threshold_ratio=0.02):
    """Select slice indices along an axis that contain meaningful signal."""
    axis = int(axis)
    axis = max(0, min(axis, volume.ndim - 1))
    reduce_axes = tuple(i for i in range(volume.ndim) if i != axis)
    slice_sums = np.abs(volume).sum(axis=reduce_axes)

    if slice_sums.size == 0:
        return np.array([volume.shape[axis] // 2], dtype=int)

    max_sum = slice_sums.max()
    if max_sum <= 0:
        return np.array([volume.shape[axis] // 2], dtype=int)

    threshold = max_sum * threshold_ratio
    content_indices = np.where(slice_sums > threshold)[0]

    if content_indices.size == 0:
        content_indices = np.arange(volume.shape[axis])

    if content_indices.size >= n_indices:
        positions = np.linspace(0, content_indices.size - 1, n_indices, dtype=int)
        return content_indices[positions]

    return content_indices.astype(int)


def choose_coordinates(volume, n_coords):
    """Choose (z, y, x) coordinates for visualization across orientations."""
    n_coords = max(1, int(n_coords))
    z_candidates = select_content_indices(volume, 0, n_coords)
    y_candidates = select_content_indices(volume, 1, n_coords)
    x_candidates = select_content_indices(volume, 2, n_coords)

    count = max(1, min(len(z_candidates), len(y_candidates), len(x_candidates), n_coords))
    coords = []
    for i in range(count):
        z_idx = int(z_candidates[min(i, len(z_candidates) - 1)])
        y_idx = int(y_candidates[min(i, len(y_candidates) - 1)])
        x_idx = int(x_candidates[min(i, len(x_candidates) - 1)])
        coords.append((z_idx, y_idx, x_idx))

    if not coords:
        center = tuple(dim // 2 for dim in volume.shape)
        coords = [center]

    return coords


def compute_display_window(volume, modality="CT"):
    foreground_mask = volume > (volume.min() + 0.05 * (volume.max() - volume.min()))
    if foreground_mask.sum() > 100:
        foreground_vals = volume[foreground_mask]
        if "MR" in str(modality).upper():
            vmin, vmax = np.percentile(foreground_vals, [1, 99])
        else:
            mean_val = foreground_vals.mean()
            std_val = foreground_vals.std()
            vmin = mean_val - 2 * std_val
            vmax = mean_val + 2 * std_val
            vmin = max(vmin, foreground_vals.min())
            vmax = min(vmax, foreground_vals.max())
    else:
        vmin, vmax = volume.min(), volume.max()
    return float(vmin), float(vmax)


def plot_raw_orientations(image_vol, spacing, coords, title="", modality="CT"):
    """Plot raw slices in axial, coronal, and sagittal orientations for given coordinates."""
    # Calculate physical dimensions
    physical_dims = tuple(shape * sp for shape, sp in zip(image_vol.shape, spacing))
    volume_mm3 = np.prod(physical_dims)
    volume_cm3 = volume_mm3 / 1000  # Convert mm³ to cm³
    
    print(f"    Raw volume shape: {image_vol.shape} voxels")
    print(f"    Voxel spacing: ({spacing[0]:.2f}, {spacing[1]:.2f}, {spacing[2]:.2f}) mm (z, y, x)")
    print(f"    Physical dimensions: ({physical_dims[0]:.1f}, {physical_dims[1]:.1f}, {physical_dims[2]:.1f}) mm")
    print(f"    Physical volume: {volume_cm3:.2f} cm³ ({volume_mm3:.0f} mm³)")
    print(f"    Intensity range: [{image_vol.min():.2f}, {image_vol.max():.2f}]")
    print(f"    Selected coordinates (z, y, x): {coords}")

    vmin, vmax = compute_display_window(image_vol, modality)
    print(f"    Display window: [{vmin:.2f}, {vmax:.2f}]")

    n_coords = len(coords)
    fig, axes = plt.subplots(n_coords, 3, figsize=(16, 4 * n_coords))
    axes = np.atleast_2d(axes)

    orientation_getters = [
        ("Axial (Z)", lambda vol, z, y, x: vol[z, :, :]),
        ("Coronal (Y)", lambda vol, z, y, x: vol[:, y, :]),
        ("Sagittal (X)", lambda vol, z, y, x: vol[:, :, x]),
    ]

    for row_idx, (z_idx, y_idx, x_idx) in enumerate(coords):
        for col_idx, (orientation_name, getter) in enumerate(orientation_getters):
            slice_img = getter(image_vol, z_idx, y_idx, x_idx)
            ax = axes[row_idx, col_idx]
            ax.imshow(slice_img, cmap='gray', vmin=vmin, vmax=vmax)
            ax.set_title(f"{orientation_name} | (z={z_idx}, y={y_idx}, x={x_idx})")
            ax.axis('off')

    spacing_str = f"({spacing[0]:.2f}, {spacing[1]:.2f}, {spacing[2]:.2f}) mm"
    phys_dims_str = f"({physical_dims[0]:.1f}, {physical_dims[1]:.1f}, {physical_dims[2]:.1f}) mm"
    fig.suptitle(
        f"{title}\n[RAW - No Preprocessing | {modality} | Shape: {image_vol.shape} | Spacing: {spacing_str} | Physical: {phys_dims_str}]",
        fontsize=14,
        y=0.995,
    )
    plt.tight_layout()
    return fig

## Process and Display Images

In [ ]:
for dataset_name, dataset_config in config['datasets'].items():
    print("\n" + "=" * 60)
    print(f"Processing dataset: {dataset_name.upper()}")
    print("=" * 60)

    images, _ = find_dataset_images(dataset_config, sam3d_root, project_root)

    if len(images) == 0:
        print(f"  ⚠️ No images found for {dataset_name}")
        continue

    sample_count = min(n_samples_per_dataset, len(images))
    if sample_count == 0:
        print("  ⚠️ Sample count is 0; skipping dataset")
        continue

    print(f"Sampling {sample_count} image(s) for visualization")
    sample_indices = sorted(random.sample(range(len(images)), sample_count))

    modality_hint = dataset_config.get('modality', '').upper()

    for sample_rank, idx in enumerate(sample_indices, start=1):
        img_path = images[idx]
        dataset_index = idx + 1  # 1-based index within the dataset

        print(f"\n📁 Processing: {img_path.name}")
        print(f"  Sample #{sample_rank}/{sample_count}")
        print(f"  Dataset index: {dataset_index}")

        try:
            print("  Loading RAW volume (no preprocessing)...")
            raw_volume, spacing = load_raw_volume(img_path)

            modality = modality_hint or ("MR" if "_MR" in img_path.name.upper() else "CT")

            coords = choose_coordinates(raw_volume, n_coords_per_sample)

            print(f"  Plotting RAW slices in three orientations [Modality: {modality}]...")
            fig = plot_raw_orientations(
                raw_volume,
                spacing,
                coords,
                title=f"{dataset_name.upper()} #{sample_rank}: {img_path.stem} (idx {dataset_index})",
                modality=modality,
            )
            plt.show()

        except Exception as e:
            print(f"  ❌ Error processing {img_path.name}: {e}")
            import traceback
            traceback.print_exc()

## Summary

✅ **All images displayed WITHOUT any preprocessing:**
- Original resolution preserved
- Original orientation preserved  
- No normalization
- No resizing or padding

Compare these with SAM_Visualization_Fixed.ipynb to see the effect of preprocessing!